# RoPE at arbitrary positions in JAX

Rotary position embeddings rotate pairs of query/key features by an angle proportional to the token's position. In a serving system positions are not always `0 .. T-1`: a decode step sits at the end of a cache, and packed sequences restart at 0 in the middle of a row. So the positions come in as data.

Write `apply_rope(x, positions, base)`:

- `x`: `(B, T, H, d)` float32, `d` even.
- `positions`: `(B, T)` int, the position of each token.
- `base`: float (10000.0 in most models).

Use the **half-split** layout (as in GPT-NeoX and Llama): feature $i$ is paired with feature $i + d/2$ for $i < d/2$. With frequencies $\omega_i = \text{base}^{-2i/d}$ and angle $\theta = p\, \omega_i$ for a token at position $p$:

$$y_i = x_i \cos\theta - x_{i + d/2} \sin\theta, \qquad y_{i + d/2} = x_{i + d/2} \cos\theta + x_i \sin\theta$$

The same rotation applies to every head. Return `y` with the shape of `x`, float32.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def apply_rope(x, positions, base):
    """x (B, T, H, d), positions (B, T). Return rotated x."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[1.0, 0.0]]]], "dtype": "float64"}, {"__nd__": [[1]], "dtype": "int64"}, 10000.0], "e": {"__nd__": [[[[0.5403022766113281, 0.8414709568023682]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[1.0, 2.0, 3.0, 4.0]], [[1.0, 2.0, 3.0, 4.0]]]], "dtype": "float64"}, {"__nd__": [[0, 2]], "dtype": "int64"}, 100.0], "e": {"__nd__": [[[[1.0, 2.0, 3.0, 4.0]], [[-3.1440389156341553, 1.1654558181762695, -0.3391430974006653, 4.317605018615723]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.3540396841506309, -0.004965368265851146, -0.5315980204626579, -2.27656646738474, 0.018715377503071538, 0.9273688169889243, 1.04333471223595, -0.5361980682592878], [2.2293638251358496, 1.9436919725173452, -0.2282261494011879, -0.15518113263191194, 0.9589650032946045, -0.2547654546817906, 0.22476604786943014, 1.2198602161614789], [1.189720511179412, -0.5298005225309926, -0.4556210442596798, -0.5490484899641733, -0.014158213733149905, 0.07491487644629682, -0.3886871673952396, 0.7746939773858059]], [[0.04961166685080974, 0.013364802881392435, -1.9687560808282676, -1.3848386724173019, -0.6952254251309311, 0.3604558900085395, -0.8661642515899808, 1.559258571445595], [0.14763065225772823, 0.2571032861402623, 0.1199626794847736, 0.4926233230275515, 0.9239783608193567, -0.5509024272675509, 1.5685336676461867, -0.7791051049437804], [0.5174087168164516, -0.20632348156510463, -0.16641061421632186, 0.6701483778463362, 0.20181373342498857, 1.6074200032750119, -0.5078405440907392, 0.5163087627818853]], [[0.5838730930775634, 2.1121054998208346, 1.6093174391556833, 0.7045759857576647, 1.6725192258085841, 0.8497750841524744, -0.7818255157008109, 0.33119384393163476], [1.227485959152611, -0.5220961113147181, 1.5266260054596967, 0.35555790705890916, -0.6288846266806212, 0.46295922718201793, -1.5698195128766215, -0.6888920698507784], [-1.0452707205120946, 0.49902375919967346, -1.023240343333598, -1.564364570748956, 0.30398989800332243, -0.4797152597893526, -0.7249728556054829, -0.5960752834847165]], [[1.0466201734360112, 0.1715199048852369, -0.7508374432014661, 2.802191825660094, 0.4890519444162596, 1.9148703924731834, 1.0518056762151695, 1.366146289456645], [-0.8759603141970402, -0.5707127573516251, 0.6441810001727681, -0.08400139404617617, 0.07818210748558983, 1.0882973630235406, -2.3025247930851296, -0.3328811763022821], [1.5363519294637775, -0.4673171070238983, 0.041992303204161155, 0.38256312625742755, 0.31958617786344307, 0.31618644120889516, -0.9325361463811137, 0.33399171339768363]], [[-0.602237443375247, 1.5785244939153444, -0.6262742446348667, 0.25522243799622374, -0.19937099369709496, 1.130903263293041, 0.12103378053967322, 0.7092786454983112], [-0.5647659329909674, -0.3806761889938276, -0.9458063186005585, -0.17089587113955884, -0.2827983177543794, -1.1331129242121305, -1.2564169602676096, 0.15964547558969563], [-0.17118107075927452, -0.7068636312580265, -0.9205985499460365, -1.1079230421515438, 0.09720268613504214, -0.5341761055420519, -0.6368553916681325, -0.5634127292008303]]], [[[-0.4736552125360425, -0.30974244600201356, 0.08450109157128341, -1.7590268375039713, -0.26072657659447357, -1.2026110351196775, -0.3739290581687834, 1.2447281409127704], [-1.9155040235679937, -1.292410234839239, -0.5201739369575917, -2.2988616982653967, 2.385126952243134, -0.6351038724703052, 1.6142547086707912, -0.0868650763080445], [1.1845518699437108, 0.8083278742845112, -0.3092626617007144, 0.49812482510587974, -0.66247863612088, -0.48188630027727875, 1.4860425949661098, -0.8679636283330758]], [[-0.9139831259420692, 2.555921717240161, -0.3870872219959422, 0.3001704587592878, 1.6387117406856069, -0.48802181482552254, -0.5634098574599227, 1.8272322026730894], [-0.02150280438129188, 0.2650441808022512, -0.33137745996055884, 0.529754029149427, -1.5881390785605611, 0.16812773168946157, 1.0017032746325099, 0.956801207213286], [-0.5126004004939266, 0.6357007386673099, 2.1370932680737873, 0.12329710286963186, 0.6756783421469383, -0.3551360587548742, -1.731340651200614, 0.7429987937643826]], [[-0.9557259297220158, -0.6988802016476103, 0.0584779298430523, -0.018708764383634197, -0.12451102697758105, 1.1128121805839744, 0.1362000178845666, 0.04067868434202199], [-1.0799535241821707, 0.9090852055306673, -1.2787490765489131, -2.269146079521028, -0.8147103520802578, 0.7023180730608086, 0.3481900112411395, 1.2712628509707942], [0.4011626052109944, 0.37109931618431297, 0.33000729588161104, -0.6417974256253315, -0.46001850480298323, -0.4583461731369784, 0.1830759458821705, -0.9317044859931699]], [[-0.42222202486715676, 0.27634254968484084, -0.07804333270551765, -0.28068464289011696, -0.20696937818217478, -1.2353850232037318, -1.3597756869302537, -0.8045038061207221], [0.9471725339217494, 0.7827091127930011, 0.2702689444069532, 0.616187774192549, 0.7862415356311107, -2.3686881696580784, -0.5535839717032988, 0.8344891908913972], [1.7596526505367276, 0.17959006982930398, 1.724191521667953, 1.160823659574389, 0.14011635188676083, -1.0495783064638748, -1.0097186586554114, -0.058459689096036056]], [[0.19172205047106686, -0.07185534909255721, 0.6091925801265499, 0.5702230069167668, 0.044031096296146745, 0.43691688006331564, 0.29647678686115725, -1.8856059016570288], [-1.2806739116798018, 0.31277362946223664, 0.7626791933408243, 0.09485397053239812, -0.2959518161659941, 0.2099805813467306, 0.45782030007796753, 0.1752653790892617], [-1.9810163679485508, 0.3211463707552701, 0.73521873654347, 0.11781525148310629, 2.2046494529738454, -1.418160682168752, -1.8082021553228729, 3.370023181831863]]]], "dtype": "float64"}, {"__nd__": [[0, 1, 2, 3, 4], [0, 1, 2, 3, 4]], "dtype": "int64"}, 10000.0], "e": {"__nd__": [[[[0.3540396988391876, -0.004965368192642927, -0.5315980315208435, -2.276566505432129, 0.018715377897024155, 0.9273688197135925, 1.043334722518921, -0.536198079586029], [2.2293639183044434, 1.9436919689178467, -0.22822615504264832, -0.1551811397075653, 0.958965003490448, -0.25476545095443726, 0.22476604580879211, 1.2198601961135864], [1.1897205114364624, -0.529800534248352, -0.45562103390693665, -0.5490484833717346, -0.014158213511109352, 0.07491488009691238, -0.3886871635913849, 0.7746939659118652]], [[0.6118173003196716, -0.022687509655952454, -1.9599961042404175, -1.3863972425460815, -0.3338851034641266, 0.35998937487602234, -0.8858081698417664, 1.557873010635376], [-0.6977357864379883, 0.31081730127334595, 0.10427160561084747, 0.4934021830558777, 0.6234545111656189, -0.5224827527999878, 1.5696548223495483, -0.7786121368408203], [0.10973672568798065, -0.3657669425010681, -0.16132397949695587, 0.6696317791938782, 0.544424831867218, 1.578791618347168, -0.509479284286499, 0.5169786810874939]], [[-1.7637943029403687, 1.9011800289154053, 1.6246310472488403, 0.7039121985435486, -0.16509926319122314, 1.2524467706680298, -0.7494850158691406, 0.33260229229927063], [0.06102871894836426, -0.603664755821228, 1.5577149391174316, 0.3569349944591522, 1.377858281135559, 0.3500064015388489, -1.5389752388000488, -0.6881795525550842], [0.1585688591003418, 0.5843812227249146, -1.0085372924804688, -1.563169240951538, -1.076966404914856, -0.37101221084594727, -0.7452913522720337, -0.5992028117179871]], [[-1.1051610708236694, -0.40202367305755615, -0.7820490598678589, 2.7980806827545166, -0.3364587128162384, 1.8800331354141235, 1.028810739517212, 1.374546766281128], [0.8561610579490662, -0.8668365478515625, 0.7129565477371216, -0.08300238102674484, -0.20101521909236908, 0.8710330724716187, -2.2821662425994873, -0.33313170075416565], [-1.5660768747329712, -0.5398845672607422, 0.06994529068470001, 0.38155943155288696, -0.09957793354988098, 0.16396279633045197, -0.9308569431304932, 0.33513790369033813]], [[0.2427641898393631, 1.0135228633880615, -0.6306133270263672, 0.252383291721344, 0.5860924124717712, 1.6563372611999512, 0.09589268267154694, 0.7102938890457153], [0.1551331728696823, 0.090628981590271, -0.8948065042495728, -0.17153307795524597, 0.6122655868530273, -1.191908359527588, -1.2932339906692505, 0.15896062552928925], [0.18545465171337128, -0.44304654002189636, -0.8943948149681091, -1.1056605577468872, 0.0660143494606018, -0.7672744393348694, -0.6731600761413574, -0.5678399205207825]]], [[[-0.47365522384643555, -0.30974245071411133, 0.08450109511613846, -1.7590268850326538, -0.26072657108306885, -1.2026110887527466, -0.37392905354499817, 1.2447280883789062], [-1.915503978729248, -1.2924102544784546, -0.5201739072799683, -2.2988617420196533, 2.385127067565918, -0.6351038813591003, 1.6142547130584717, -0.08686507493257523], [1.1845518350601196, 0.808327853679657, -0.3092626631259918, 0.4981248378753662, -0.6624786257743835, -0.4818862974643707, 1.4860426187515259, -0.8679636120796204]], [[-1.872755527496338, 2.5918736457824707, -0.3814338445663452, 0.29834306240081787, 0.11630940437316895, -0.23041734099388123, -0.5672524571418762, 1.8275315761566162], [1.3247548341751099, 0.24693529307842255, -0.34137773513793945, 0.5287970304489136, -0.8761691451072693, 0.19374805688858032, 0.9983394742012024, 0.9573305249214172], [-0.8455228805541992, 0.66797935962677, 2.15429949760437, 0.12255404144525528, -0.06626784801483154, -0.2898976802825928, -1.7098835706710815, 0.7431217432022095]], [[0.51093989610672, -0.9060307741165161, 0.05574242025613785, -0.01879008300602436, -0.8172242641448975, 0.9517840147018433, 0.13734225928783417, 0.0406411848962307], [1.1902332305908203, 0.7514349818229675, -1.2854567766189575, -2.271683931350708, -0.6429598331451416, 0.8689258098602295, 0.3225471079349518, 1.2667219638824463], [0.25135111808776855, 0.4547613859176636, 0.32628002762794495, -0.6399326920509338, 0.5562113523483276, -0.3754836916923523, 0.18963904678821564, -0.9329861402511597]], [[0.4472041726112366, 0.6290813684463501, -0.037221070379018784, -0.27826988697052, 0.14531415700912476, -1.098543643951416, -1.3615047931671143, -0.8053422570228577], [-1.0486481189727783, 1.4477458000183105, 0.2867523431777954, 0.6136815547943115, -0.6447082757949829, -2.031587839126587, -0.5452280640602112, 0.8363339900970459], [-1.761816143989563, 0.4817405641078949, 1.7537027597427368, 1.1609938144683838, 0.10960803925991058, -0.9496279358863831, -0.9575463533401489, -0.05497696250677109]], [[-0.09199504554271698, -0.23632660508155823, 0.5968494415283203, 0.5777609348297119, -0.17387636005878448, 0.3744453191757202, 0.32060083746910095, -1.883310079574585], [0.6131271719932556, 0.20631329715251923, 0.7437612414360046, 0.09415214508771896, 1.1626641750335693, 0.3152046799659729, 0.4879531264305115, 0.17564338445663452], [2.9633629322052, 0.8480531573295593, 0.806939423084259, 0.10433425009250641, 0.05818307399749756, -1.181152105331421, -1.7773548364639282, 3.3704676628112793]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[-1.0000690137088297, -2.4453801820724297, -0.566224032434944, 0.827222135071063, -0.7153881007925209, -0.4227392727410965, 0.7465454922302805, 1.9446185857629645], [0.7620217988102919, 0.31547230811398624, -0.4419744070480968, 0.7531307534793433, 2.793154990748989, -1.129498677765437, -1.0726666993098672, -0.37124593537805245]], [[0.7187868409621374, 0.6176247612327831, -0.22207633881890204, -0.4678343147818037, 1.357409691958232, 0.45893314154344245, 1.3322976895833891, -0.23313095782934345], [1.143265674789309, -0.8404007761486365, 2.0613681165508844, -0.3998595001041209, -0.3578531754698278, 0.43681065220995413, -0.18594421917422793, -0.6032693016990295]], [[0.455052108224932, -0.6450958593331355, 1.9272831228844158, -0.6923062361321382, 0.828613722344223, -0.7530497179293031, -1.7399053321501032, -2.3303258803220372], [0.043196536565342024, -1.4124378975062128, 0.14361799425387786, -0.8081774126549782, -0.0854248665664205, -0.026163410317532936, 1.7792866562003042, -0.18872808445476]], [[-1.3955092836964225, -2.327373435539872, 0.8980586868367981, 0.0986320196795216, 1.4260336429328788, -1.604322190603503, -0.18285459418876454, -0.8292471892104415], [0.4915122878570852, -0.32271916335137946, 1.4067976206967492, -0.42496654794347555, -0.22048545567502253, -0.6408268445418258, 0.4702186960356857, 1.0928876091372461]]]], "dtype": "float64"}, {"__nd__": [[0, 1, 0, 1]], "dtype": "int64"}, 10000.0], "e": {"__nd__": [[[[-1.00006902217865, -2.445380210876465, -0.5662240386009216, 0.8272221088409424, -0.7153881192207336, -0.42273926734924316, 0.7465454936027527, 1.944618582725525], [0.7620217800140381, 0.3154723048210144, -0.441974401473999, 0.7531307339668274, 2.7931549549102783, -1.1294987201690674, -1.0726666450500488, -0.3712459206581116]], [[-0.7538586854934692, 0.5687223672866821, -0.23538799583911896, -0.4676009714603424, 1.3382498025894165, 0.5182999968528748, 1.3300102949142456, -0.23359869420528412], [0.9188321828842163, -0.8798105716705322, 2.063124656677246, -0.39925605058670044, 0.7686760425567627, 0.3507283329963684, -0.16532157361507416, -0.6036688685417175]], [[0.45505210757255554, -0.6450958847999573, 1.9272831678390503, -0.6923062205314636, 0.8286136984825134, -0.7530497312545776, -1.7399053573608398, -2.3303258419036865], [0.04319653660058975, -1.412437915802002, 0.14361798763275146, -0.8081774115562439, -0.08542487025260925, -0.026163410395383835, 1.7792866230010986, -0.18872807919979095]], [[-1.9539626836776733, -2.15558123588562, 0.8998423218727112, 0.09946122765541077, -0.40379130840301514, -1.8286569118499756, -0.17386500537395477, -0.8291481137275696], [0.451097309589386, -0.2571309804916382, 1.4020252227783203, -0.42605921626091003, 0.2944645285606384, -0.6698434948921204, 0.4842629134654999, 1.092462182044983]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.5420536442248179, -0.6172525111912264, -1.0557209891136248, -0.9116841941241273, 0.18325677831132295, -0.5024521824672545, -0.1653632617870115, 1.1267548399449159, 0.4745349134301127, 0.22323933664835047, -0.4972657790124328, -0.19358817099999884, -0.29441439055535024, 0.044412448248318886, -0.20884257207760334, 0.17016347886414396], [-0.6398335327550475, 1.298880568548161, 0.38487987557375164, 0.30609635043370326, 0.9028976392528592, -1.0678546199537908, -1.1880679760957022, -1.4888833512354818, -0.7590063246044053, 0.5647461253844039, 0.18287631390182438, 1.576467479632227, -1.4075137998684015, 0.4419233728367722, -1.2741434167862464, -1.278116868999641], [-1.2705424555307891, 1.2005269681783062, 0.07562601958032478, 1.2620675564196968, 0.0873031578324804, 0.6998676951007154, 0.6604934472505254, 0.5495834985709621, 0.1465220563334789, 0.9552068820468754, 0.19037619235790756, 0.07814387145746318, 0.042409416360465226, -0.5795018434504766, -1.5187849816001224, -0.7634604089523968], [1.7860276985331929, 0.15366393549667254, -1.0187214376338747, -0.26527005692315664, 0.9750909452813495, 1.8527714923370258, -1.8617493720649188, -1.614904083795972, -2.145078984397006, -0.6131197548233105, -0.9310182327710218, 1.1536863171751823, 0.6548707670216672, -0.5611751406538557, 2.312537542776966, -0.6124496127596035]]], [[[0.32174056535458734, 0.7631548889759052, -1.0090384581693503, 0.1275425701535049, 0.27510309229291524, -0.1675858691818282, 0.7187221671634141, -0.1655426253125277, -0.648960171034972, 0.8466534655256985, -0.4958118895446977, 0.4629523629599924, -0.4913921142632256, 0.587965235715295, 0.14372812681033814, -1.2071354612358436], [0.4148422407084458, -0.43263543225271606, 1.7708758009328895, -0.5388532355537631, 0.8409868467772856, -2.269504419420257, -0.5079391020857246, -1.7945074727449966, 0.04786134469819158, -0.8068141664651547, 1.3793754410911117, 0.7011934278415044, 1.482794091132957, -0.9185391859534852, -2.236912057546303, -0.027369191766853956], [1.4111081967506844, 0.2959430395026134, 1.911644215117884, -0.25143091552955643, -0.6657519980069967, 0.5412935143175563, 1.096906933003817, -0.3772937887072629, -0.6397460820747768, -0.6674967515428102, 1.9852591497080092, -2.0868496711576383, 1.207563778480383, -0.12540675377957253, 1.590458284083731, 0.8254867928063687], [-1.134246255496575, 0.2409608741302857, -0.4806449722588052, -0.5433575387106238, -1.3876899996484502, 1.293394694458536, -1.901886350682376, 0.2457005986971033, 2.06956482228666, 0.523731301817298, 1.6468506071696942, 0.9246980704996453, 1.332842380312345, -0.8571843853305717, 0.18057028956418614, 0.5059074522871886]]]], "dtype": "float64"}, {"__nd__": [[37], [4095]], "dtype": "int64"}, 10000.0], "e": {"__nd__": [[[[0.7202767729759216, -0.22988833487033844, 0.6318876147270203, -0.17740969359874725, 0.27732008695602417, -0.5042014122009277, -0.15752467513084412, 1.12468683719635, 0.014383494853973389, 0.61480712890625, 0.9810903072357178, -0.9149701595306396, -0.20822212100028992, -0.014546215534210205, -0.21481667459011078, 0.1833350509405136], [-0.978187084197998, 1.2717440128326416, -0.22952215373516083, -1.3321477174758911, 1.3507750034332275, -1.112142562866211, -1.140122413635254, -1.4738272428512573, -0.16919687390327454, -0.6234550476074219, -0.3590207099914551, 0.8968430757522583, -0.9857618808746338, 0.3142431974411011, -1.3172199726104736, -1.2954494953155518], [-0.8781985640525818, 1.5054352283477783, 0.03672976791858673, 0.42039594054222107, 0.06605921685695648, 0.7627321481704712, 0.7162235975265503, 0.5584784746170044, 0.9297925233840942, -0.29555046558380127, -0.2015274465084076, 1.1925554275512695, 0.07110962271690369, -0.493838906288147, -1.4933128356933594, -0.7569779753684998], [-0.013389348983764648, -0.367455393075943, 0.3706905245780945, -1.1657617092132568, 0.6722925305366516, 1.905613899230957, -1.9460195302963257, -1.6076277494430542, -2.791250467300415, -0.5143004655838013, 1.3293520212173462, 0.205815389752388, 0.963161826133728, -0.341050386428833, 2.2420854568481445, -0.6313022971153259]]], [[[-0.6687734127044678, 0.13318419456481934, -0.023743927478790283, 0.196306973695755, -0.32706204056739807, -0.375244677066803, -0.29887905716896057, 1.1164135932922363, -0.27822375297546387, 1.1320289373397827, -1.1240215301513672, -0.43824148178100586, 0.45845210552215576, 0.4826795756816864, -0.6692463159561157, -0.4880584180355072], [0.0203874371945858, 0.11346274614334106, -0.4105103015899658, 0.8619183301925659, -0.6742273569107056, -1.7615456581115723, -1.5299122333526611, -0.4624154567718506, -0.4170961081981659, -0.9084320068359375, 2.206843614578247, -0.1978175938129425, -1.5656801462173462, -1.7003886699676514, 1.7091360092163086, -1.7341217994689941], [-0.7314515113830566, 0.6273836493492126, -0.8837946653366089, -1.134390950202942, 0.7934991121292114, 0.5489272475242615, 0.6618403196334839, -0.8970391750335693, -1.365825891494751, -0.37352901697158813, 2.6104683876037598, 1.7695525884628296, -1.1277389526367188, 0.08604894578456879, -1.8151390552520752, -0.13820187747478485], [2.1398885250091553, -0.10619014501571655, -1.6834173202514648, 1.0076563358306885, 1.524794578552246, 1.520067811012268, 1.2482527494430542, -0.4198632836341858, 0.9952332973480225, 0.5666394233703613, 0.3305183947086334, -0.36733168363571167, -1.173521876335144, -0.31149429082870483, 1.4462510347366333, 0.3742004632949829]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[0.25041527122262847, -0.9066606857599893, -0.9915658131915795, 0.034911947425910146, 0.4416569845266169, 0.28692033572757136]], [[-1.6218597143476525, -0.5278388873410679, -0.5540454054467021, 0.42182601114824014, -0.8593392461625653, 0.11774550354033447]], [[-1.2235044282134153, -0.3991783356913868, 0.7069233401050825, 0.8408275253396778, -1.2298738533114901, 0.11224129724061226]]]], "dtype": "float64"}, {"__nd__": [[5, 5, 5]], "dtype": "int64"}, 500000.0], "e": {"__nd__": [[[[0.10451126098632812, -0.9326664805412292, -0.9917932748794556, -0.230226069688797, 0.3837026357650757, 0.2861332297325134]], [[-0.055561065673828125, -0.47269269824028015, -0.5541386604309082, 1.6748967170715332, -0.8908644318580627, 0.11730571836233139]], [[0.45922794938087463, -0.32096055150032043, 0.7068340182304382, 1.4117591381072998, -1.2525644302368164, 0.11280235648155212]]]], "dtype": "float32"}}, {"a": [{"__nd__": [[[[-0.6764009159171281, 1.8942880277259206, 0.4744419570399953, 0.08337295805421349], [1.9416391017449228, -2.922139069851382, -0.8272198136265027, 0.6668223381641396]], [[-0.0034668171666841574, 0.4583281480306128, -0.5602060064908454, -0.2036550569132401], [0.31896388498438905, 1.246109113547895, 2.252940140990888, -1.0781754674335802]]], [[[0.6088685998015129, -2.090406998918412, -0.39988881258210107, 0.4042273492680625], [-0.7506775593311028, 0.3103683623810141, -0.08123135665121275, 1.8283738760539674]], [[0.191524033456207, -0.5462208743105064, -1.552081174483661, -0.3152469353164281], [0.829565722651191, 2.4289143283149373, 0.26351097043779453, -0.15076196600355005]]], [[[-0.02165607470947082, 0.4842352989425276, 1.1139442611292143, 1.0919073944141449], [0.25215539946861903, 0.5446321354464523, -0.10750169347308944, 1.2827207781959817]], [[-1.3906250712366302, 1.0728246747390735, 0.16350537700934484, 0.08529948822896441], [0.31906458774479773, 0.0057568224502638245, -0.7633813780626244, 0.05366409963879276]]]], "dtype": "float64"}, {"__nd__": [[0, 0], [0, 0], [0, 0]], "dtype": "int64"}, 10000.0], "e": {"__nd__": [[[[-0.676400899887085, 1.8942880630493164, 0.4744419455528259, 0.08337295800447464], [1.9416390657424927, -2.9221391677856445, -0.8272197842597961, 0.6668223142623901]], [[-0.003466817084699869, 0.45832815766334534, -0.5602059960365295, -0.20365506410598755], [0.318963885307312, 1.246109127998352, 2.2529401779174805, -1.07817542552948]]], [[[0.6088685989379883, -2.090406894683838, -0.399888813495636, 0.4042273461818695], [-0.7506775856018066, 0.3103683590888977, -0.08123135566711426, 1.828373908996582]], [[0.19152402877807617, -0.5462208986282349, -1.5520812273025513, -0.31524693965911865], [0.829565703868866, 2.4289143085479736, 0.26351097226142883, -0.15076196193695068]]], [[[-0.02165607549250126, 0.4842352867126465, 1.1139442920684814, 1.0919073820114136], [0.2521553933620453, 0.5446321368217468, -0.10750169306993484, 1.2827208042144775]], [[-1.3906251192092896, 1.0728247165679932, 0.16350537538528442, 0.08529949188232422], [0.31906458735466003, 0.005756822414696217, -0.7633813619613647, 0.05366409942507744]]]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(apply_rope, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")